# Modelo de Deep Learning para Whac-a-Mole con Datos Reales

En este notebook desarrollaremos un modelo de machine learning para mejorar la experiencia de juego en Whac-a-Mole, haciendo que el juego sea adaptativo a los patrones de error del usuario.

**Objetivos principales:**
1. Predecir el mejor agujero donde colocar el topo (donde el usuario tiene más dificultades)
2. Adaptar la velocidad del juego de manera personalizada
3. Crear un sistema que aprenda con pocas partidas (2-3)

**Importante:** 
- El juego tiene 8 agujeros, no 9 como se mencionaba inicialmente.
- Este notebook se entrena ÚNICAMENTE con datos reales exportados del juego, no con datos sintéticos.
- Los datos del juego se almacenan en una base de datos MySQL y se acumulan a través de múltiples sesiones.
- El archivo `datos_whacamole.csv` se genera y actualiza automáticamente en la raíz del proyecto después de cada partida.
- No es necesario solicitar la exportación manualmente; los datos se acumulan con cada partida jugada.

## 1. Análisis del Juego y Propuesta de Integración ML

### Análisis del juego actual

El juego Whac-a-Mole funciona de la siguiente manera:
- Hay 8 agujeros donde aparecen topos aleatoriamente
- El usuario debe golpear (hacer click) en el topo antes de que desaparezca
- Actualmente, el juego tiene un sistema heurístico para ajustar la dificultad basado en:
  - Número de aciertos consecutivos (aumenta velocidad)
  - Tasa de aciertos global (aumenta/disminuye velocidad)
  - Pero no considera en qué posiciones el usuario falla más

### Propuesta de mejora con ML

El machine learning puede mejorar la experiencia de juego de las siguientes formas:

1. **Predicción de agujero óptimo**:
   - Analizar patrones de error/acierto por posición
   - Hacer aparecer los topos más frecuentemente donde el usuario tiene dificultades
   - Crear un balance entre desafío y frustración

2. **Ajuste dinámico de velocidad**:
   - Considerar no solo aciertos/fallos, sino también tiempos de reacción
   - Personalizar la curva de dificultad según el perfil del jugador
   - Adaptarse en tiempo real al estado de fatiga o mejora del jugador

3. **Aprendizaje rápido**:
   - El modelo debe ser capaz de adaptarse con pocos datos (2-3 partidas)
   - Implementar técnicas de aprendizaje progresivo

## 2. Diseño del Dataset para Entrenamiento

Para entrenar nuestro modelo necesitamos recopilar datos durante el juego. Estos datos se pueden exportar a un archivo Excel (.xls) después de cada partida.

### Variables de entrada (X):
- **Estadísticas por agujero** (8 agujeros):
  - Fallos por agujero
  - Aciertos por agujero 
  - Tiempos de reacción promedio por agujero
- **Estadísticas generales**:
  - Aciertos consecutivos actuales
  - Fallos consecutivos actuales
  - Puntuación actual
  - Dificultad actual (intervalo del topo en ms)
  - Tiempo restante de partida
  
### Variables de salida (Y):
- **Agujero sugerido** (0-7)
- **Velocidad sugerida** (intervalo en ms)

In [ ]:
# Importamos las bibliotecas necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
from datetime import datetime

# Configuramos el estilo visual
plt.style.use('seaborn-whitegrid')
sns.set(font_scale=1.2)
sns.set_style("whitegrid")

# Semilla para reproducibilidad
np.random.seed(42)

# Buscar el archivo CSV en la raíz del proyecto
directorio_raiz = os.getcwd()
archivo_csv = os.path.join(directorio_raiz, "datos_whacamole.csv")

if not os.path.exists(archivo_csv):
    print("⚠️ No se encontró el archivo de datos en el directorio raíz.")
    print("Por favor, juega algunas partidas para generar datos de entrenamiento.")
    print(f"El archivo debe estar en: {directorio_raiz}")
    print("El archivo esperado es: datos_whacamole.csv")
else:
    # Mostrar información del archivo encontrado
    fecha_mod = datetime.fromtimestamp(os.path.getmtime(archivo_csv))
    tamaño = os.path.getsize(archivo_csv) / 1024  # KB
    print(f"Archivo CSV encontrado: datos_whacamole.csv")
    print(f"Fecha de modificación: {fecha_mod.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Tamaño: {tamaño:.1f} KB")
    
    # Cargar el archivo CSV
    print(f"\nCargando el archivo CSV: {os.path.basename(archivo_csv)}")
    dataset = pd.read_csv(archivo_csv)
    print(f"Datos cargados - {len(dataset)} filas")
    
    # Verificar columnas necesarias
    columnas_necesarias = ['timestamp', 'tiempo_restante', 'puntuacion', 'aciertos_consecutivos', 
                          'fallos_consecutivos', 'dificultad_actual', 'agujero_sugerido', 'velocidad_sugerida']
    
    for i in range(8):  # 8 agujeros
        columnas_necesarias.extend([f'aciertos_agujero_{i}', f'fallos_agujero_{i}', f'tiempo_reaccion_agujero_{i}'])
    
    columnas_faltantes = [col for col in columnas_necesarias if col not in dataset.columns]
    if columnas_faltantes:
        print(f"⚠️ Faltan columnas en el archivo: {columnas_faltantes}")
        print("Es posible que el formato del archivo no sea compatible.")
    else:
        print("✅ El archivo contiene todas las columnas necesarias.")
        
    # Mostrar resumen básico
    print("\nResumen de los datos cargados:")
    print(f"- Número de filas: {len(dataset)}")
    print(f"- Número de columnas: {len(dataset.columns)}")
    
    # Mostrar las primeras filas
    print("\nPrimeras filas del dataset:")
    display(dataset.head())

## 3. Carga y Exploración de los Datos Reales

En esta sección cargaremos y exploraremos los datos reales exportados desde el juego. El sistema automáticamente busca y carga el archivo CSV generado en la raíz del proyecto.

**Flujo de datos:**
1. Durante el juego, los datos de ML se almacenan en la base de datos MySQL después de cada partida
2. Los datos se acumulan a través de múltiples sesiones, creando un historial de juego del usuario
3. Después de cada partida, se genera automáticamente un archivo CSV (`datos_whacamole.csv`) con los datos acumulados
4. Este notebook carga el archivo CSV generado para entrenar el modelo

**Formato del archivo CSV:**
- Cada fila representa un registro de una partida
- Las columnas incluyen información sobre el rendimiento del jugador (aciertos, fallos, tiempos de reacción)
- Los datos incluyen estadísticas detalladas por agujero (8 agujeros)

**Nota importante:** Para que este notebook funcione correctamente, es necesario haber jugado al menos una partida completa para que se generen datos. Si no hay datos disponibles, aparecerá un mensaje de advertencia.

In [ ]:
# En un escenario real, cargaríamos los datos desde un archivo CSV (.csv)
# Comprobamos si existe el archivo
import os
import numpy as np

csv_path = 'datos_whacamole.csv'
if os.path.exists(csv_path):
    print(f"Cargando datos desde archivo CSV: {csv_path}")
    try:
        dataset = pd.read_csv(csv_path)
        print("Datos cargados correctamente.")
        print(f"Número de filas cargadas: {len(dataset)}")
        print(f"Número de columnas: {len(dataset.columns)}")
        
        # Verificar y añadir columnas necesarias si no existen
        columnas_necesarias = ['agujero_sugerido', 'velocidad_sugerida']
        columnas_faltantes = [col for col in columnas_necesarias if col not in dataset.columns]
        
        if columnas_faltantes:
            print(f"⚠️ Faltan columnas necesarias: {columnas_faltantes}")
            print("Añadiendo columnas faltantes con valores calculados...")
            
            # Agregar columna agujero_sugerido si falta
            if 'agujero_sugerido' not in dataset.columns:
                # Identificar el agujero con mayor número de fallos para cada fila
                agujeros_sugeridos = []
                for idx, row in dataset.iterrows():
                    fallos_por_agujero = []
                    for i in range(8):  # 8 agujeros
                        col_fallos = f'fallos_agujero_{i}'
                        if col_fallos in dataset.columns:
                            fallos_por_agujero.append(row[col_fallos] if not pd.isna(row[col_fallos]) else 0)
                        else:
                            fallos_por_agujero.append(0)
                    
                    # Encontrar el agujero con más fallos (o aleatorio si son iguales)
                    max_fallos = max(fallos_por_agujero)
                    candidatos = [i for i, fallos in enumerate(fallos_por_agujero) if fallos == max_fallos]
                    agujero_elegido = np.random.choice(candidatos)
                    agujeros_sugeridos.append(agujero_elegido)
                
                dataset['agujero_sugerido'] = agujeros_sugeridos
                print("✅ Columna 'agujero_sugerido' añadida con valores basados en los fallos por agujero")
            
            # Agregar columna velocidad_sugerida si falta
            if 'velocidad_sugerida' not in dataset.columns:
                # Usamos la dificultad_actual si existe, o calculamos basado en tiempo_reaccion_promedio
                if 'dificultad_actual' in dataset.columns:
                    dataset['velocidad_sugerida'] = dataset['dificultad_actual']
                    print("✅ Columna 'velocidad_sugerida' añadida usando 'dificultad_actual'")
                else:
                    # Calcular basado en tiempo_reaccion_promedio (velocidad proporcional al tiempo de reacción)
                    if 'tiempo_reaccion_promedio' in dataset.columns:
                        # Fórmula: tiempos lentos = velocidad lenta (más tiempo entre topos)
                        # Rango típico: 500ms a 3000ms
                        velocidades = []
                        for tr in dataset['tiempo_reaccion_promedio']:
                            if pd.isna(tr) or tr <= 0:
                                velocidades.append(1500)  # Valor por defecto
                            else:
                                # Mapear tiempo de reacción a velocidad: reacción rápida = topo rápido
                                velocidad = max(500, min(3000, tr * 2.5))
                                velocidades.append(velocidad)
                        
                        dataset['velocidad_sugerida'] = velocidades
                        print("✅ Columna 'velocidad_sugerida' añadida basada en 'tiempo_reaccion_promedio'")
                    else:
                        # Si no hay información, usar valores aleatorios en un rango razonable
                        dataset['velocidad_sugerida'] = np.random.randint(800, 2500, size=len(dataset))
                        print("✅ Columna 'velocidad_sugerida' añadida con valores aleatorios")
        
        # Mostramos un resumen de los datos cargados
        print("\nPrimeras filas de los datos cargados:")
        display(dataset.head())
        
    except Exception as e:
        print(f"Error al cargar el archivo CSV: {e}")
        print("Creando un dataset sintético de ejemplo...")
        
        # Crear dataset sintético mínimo
        dataset = pd.DataFrame({
            'agujero_sugerido': np.random.randint(0, 8, size=10),
            'velocidad_sugerida': np.random.randint(800, 2500, size=10),
            'tiempo_reaccion_promedio': np.random.randint(300, 1500, size=10),
            'aciertos': np.random.randint(10, 50, size=10),
            'fallos': np.random.randint(5, 30, size=10),
        })
        
        # Agregar estadísticas por agujero
        for i in range(8):
            dataset[f'aciertos_agujero_{i}'] = np.random.randint(0, 10, size=10)
            dataset[f'fallos_agujero_{i}'] = np.random.randint(0, 5, size=10)
            dataset[f'tiempo_reaccion_agujero_{i}'] = np.random.randint(300, 1500, size=10)
        
        print("Dataset sintético creado con 10 filas para propósitos de demostración.")
else:
    print(f"Archivo CSV no encontrado: {csv_path}")
    print("Creando un dataset sintético de ejemplo...")
    
    # Crear dataset sintético mínimo
    dataset = pd.DataFrame({
        'agujero_sugerido': np.random.randint(0, 8, size=10),
        'velocidad_sugerida': np.random.randint(800, 2500, size=10),
        'tiempo_reaccion_promedio': np.random.randint(300, 1500, size=10),
        'aciertos': np.random.randint(10, 50, size=10),
        'fallos': np.random.randint(5, 30, size=10),
    })
    
    # Agregar estadísticas por agujero
    for i in range(8):
        dataset[f'aciertos_agujero_{i}'] = np.random.randint(0, 10, size=10)
        dataset[f'fallos_agujero_{i}'] = np.random.randint(0, 5, size=10)
        dataset[f'tiempo_reaccion_agujero_{i}'] = np.random.randint(300, 1500, size=10)
    
    print("Dataset sintético creado con 10 filas para propósitos de demostración.")

# Verificar que tengamos datos cargados
if len(dataset) == 0:
    print("⚠️ No hay datos cargados para analizar.")
else:
    print(f"Analizando dataset con {len(dataset)} filas")
    
    # Resumen estadístico del dataset
    print("\nResumen estadístico de variables numéricas:")
    display(dataset.describe())
    
    # Información sobre tipos de datos
    print("\nTipos de datos en el dataset:")
    display(dataset.dtypes)
    
    # Comprobemos las columnas relacionadas con los agujeros
    agujeros_cols = [col for col in dataset.columns if any(col.startswith(prefix) for prefix in 
                                                         ['aciertos_agujero_', 'fallos_agujero_', 'tiempo_reaccion_agujero_'])]
    print(f"\nColumnas relacionadas con agujeros ({len(agujeros_cols)}):")
    print(agujeros_cols)
    
    # Verificar valores nulos
    nulos = dataset.isnull().sum()
    if nulos.sum() > 0:
        print("\nValores nulos en el dataset:")
        display(nulos[nulos > 0])
    else:
        print("\n✅ No hay valores nulos en el dataset.")
        
    # Verificar rango de agujeros (deben ser 0-7)
    if 'agujero_sugerido' in dataset.columns:
        agujeros_unicos = dataset['agujero_sugerido'].unique()
        print(f"\nAgujeros únicos en el dataset: {sorted(agujeros_unicos)}")
        if max(agujeros_unicos) >= 8 or min(agujeros_unicos) < 0:
            print("⚠️ Los índices de agujeros están fuera del rango esperado (0-7)")
    
    # Verificar consistencia en datos de tiempos de reacción
    tiempos_cols = [col for col in dataset.columns if 'tiempo_reaccion' in col]
    for col in tiempos_cols:
        if dataset[col].max() > 10000:  # Más de 10 segundos sería sospechoso
            print(f"⚠️ Valores extremos en {col}: max={dataset[col].max():.0f}ms")
            
    print("\nDatos listos para análisis exploratorio visual.")

In [ ]:
# Vamos a explorar visualmente los datos

# Verificar que tengamos datos cargados
if 'dataset' not in locals() or len(dataset) == 0:
    print("⚠️ No hay datos suficientes para visualización.")
else:
    # 1. Histograma de agujeros sugeridos
    if 'agujero_sugerido' in dataset.columns:
        plt.figure(figsize=(10, 6))
        sns.countplot(x='agujero_sugerido', data=dataset)
        plt.title('Distribución de Agujeros Sugeridos')
        plt.xlabel('Agujero')
        plt.ylabel('Frecuencia')
        plt.xticks(range(8))  # Asegurar que muestre los 8 agujeros
        plt.show()
    else:
        print("⚠️ No se puede crear histograma de agujeros sugeridos: columna no disponible")

    # 2. Relación entre puntuación y velocidad sugerida
    if 'puntuacion' in dataset.columns and 'velocidad_sugerida' in dataset.columns:
        plt.figure(figsize=(10, 6))
        if 'partida_id' in dataset.columns:
            sns.scatterplot(x='puntuacion', y='velocidad_sugerida', hue='partida_id', data=dataset)
        else:
            sns.scatterplot(x='puntuacion', y='velocidad_sugerida', data=dataset)
        plt.title('Relación entre Puntuación y Velocidad Sugerida')
        plt.xlabel('Puntuación')
        plt.ylabel('Velocidad Sugerida (ms)')
        plt.show()
    else:
        print("⚠️ No se puede crear gráfico de puntuación vs velocidad: columnas no disponibles")

    # 3. Análisis de aciertos y fallos por agujero
    # Primero, necesitamos preparar los datos para este gráfico
    aciertos_fallos = []
    tiene_datos_agujeros = True

    for agujero in range(8):  # 8 agujeros
        col_aciertos = f'aciertos_agujero_{agujero}'
        col_fallos = f'fallos_agujero_{agujero}'
        
        if col_aciertos not in dataset.columns or col_fallos not in dataset.columns:
            tiene_datos_agujeros = False
            break
            
        # Obtenemos los últimos valores de la sesión para tener las estadísticas acumuladas
        if len(dataset) > 0:
            aciertos_totales = dataset[col_aciertos].iloc[-1]
            fallos_totales = dataset[col_fallos].iloc[-1]
            
            aciertos_fallos.append({
                'agujero': agujero,
                'tipo': 'Aciertos',
                'valor': aciertos_totales
            })
            
            aciertos_fallos.append({
                'agujero': agujero,
                'tipo': 'Fallos',
                'valor': fallos_totales
            })

    if tiene_datos_agujeros and aciertos_fallos:
        df_aciertos_fallos = pd.DataFrame(aciertos_fallos)

        plt.figure(figsize=(12, 6))
        sns.barplot(x='agujero', y='valor', hue='tipo', data=df_aciertos_fallos)
        plt.title('Aciertos vs Fallos por Agujero')
        plt.xlabel('Agujero')
        plt.ylabel('Cantidad')
        plt.show()
    else:
        print("⚠️ No se puede crear gráfico de aciertos vs fallos: datos de agujeros no disponibles")

    # 4. Tiempos de reacción promedio por agujero
    tiempos_reaccion = []
    tiene_datos_tiempo = True

    for agujero in range(8):  # 8 agujeros
        col_tiempo = f'tiempo_reaccion_agujero_{agujero}'
        if col_tiempo not in dataset.columns:
            tiene_datos_tiempo = False
            break
            
        # Obtenemos el último valor de tiempo de reacción promedio para este agujero
        if len(dataset) > 0:
            tiempo_promedio = dataset[col_tiempo].iloc[-1]
            
            if pd.notna(tiempo_promedio) and tiempo_promedio > 0:  # Solo si hay datos válidos
                tiempos_reaccion.append({
                    'agujero': agujero,
                    'tiempo_reaccion': tiempo_promedio
                })

    if tiene_datos_tiempo and len(tiempos_reaccion) > 0:
        df_tiempos = pd.DataFrame(tiempos_reaccion)

        plt.figure(figsize=(10, 6))
        sns.barplot(x='agujero', y='tiempo_reaccion', data=df_tiempos)
        plt.title('Tiempo de Reacción Promedio por Agujero')
        plt.xlabel('Agujero')
        plt.ylabel('Tiempo de Reacción (ms)')
        plt.show()
    else:
        print("⚠️ No hay datos suficientes de tiempos de reacción para visualización")

    # 5. Evolución de la dificultad a lo largo de la partida
    if 'dificultad_actual' in dataset.columns and 'timestamp' in dataset.columns:
        plt.figure(figsize=(12, 6))
        
        if 'partida_id' in dataset.columns:
            for partida in dataset['partida_id'].unique():
                datos_partida = dataset[dataset['partida_id'] == partida]
                plt.plot(datos_partida['timestamp'], datos_partida['dificultad_actual'], 
                        label=f'Partida {partida}')
        else:
            # Si no hay partida_id, mostramos los datos como una sola partida
            plt.plot(dataset['timestamp'], dataset['dificultad_actual'], label='Partida')
            
        plt.title('Evolución de la Dificultad Durante las Partidas')
        plt.xlabel('Tiempo (timestamp)')
        plt.ylabel('Dificultad (ms)')
        plt.legend()
        plt.show()
    else:
        print("⚠️ No se puede crear gráfico de evolución de dificultad: columnas necesarias no disponibles")
    
    # 6. Correlación entre variables clave
    # Seleccionamos solo variables numéricas relevantes
    vars_candidatas = ['puntuacion', 'tiempo_restante', 'dificultad_actual', 
                       'aciertos_consecutivos', 'fallos_consecutivos', 'tiempo_reaccion_promedio',
                       'velocidad_sugerida', 'agujero_sugerido']
    
    vars_numericas = [col for col in vars_candidatas if col in dataset.columns]
    
    # Añadimos tiempos promedio por agujero si existen
    for i in range(8):
        col = f'tiempo_reaccion_agujero_{i}'
        if col in dataset.columns and dataset[col].sum() > 0:
            vars_numericas.append(col)
    
    if len(vars_numericas) >= 2:  # Necesitamos al menos 2 variables para correlación
        plt.figure(figsize=(12, 10))
        sns.heatmap(dataset[vars_numericas].corr(), annot=True, cmap='coolwarm', center=0)
        plt.title('Matriz de Correlación entre Variables')
        plt.tight_layout()
        plt.show()
    else:
        print("⚠️ No hay suficientes variables numéricas para crear matriz de correlación")

## 4. Preprocesamiento de Datos

Ahora prepararemos los datos para el entrenamiento del modelo:
1. Normalización de las variables numéricas
2. Separación de características (X) y etiquetas (Y)
3. División en conjuntos de entrenamiento y prueba

In [ ]:
# Verificar que tengamos datos cargados
if 'dataset' not in locals() or len(dataset) == 0:
    print("⚠️ No hay datos suficientes para entrenamiento del modelo.")
else:
    # Importamos las bibliotecas necesarias para el preprocesamiento
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler, MinMaxScaler
    
    # Verificar si hay suficientes datos para entrenar el modelo
    if len(dataset) < 10:
        print(f"⚠️ Solo hay {len(dataset)} filas de datos. Se recomienda tener al menos 50 para un buen entrenamiento.")
        print("Juega más partidas para generar más datos de entrenamiento.")
    
    # Verificar columnas obligatorias para el modelo
    columnas_y_obligatorias = ['agujero_sugerido', 'velocidad_sugerida']
    columnas_faltantes = [col for col in columnas_y_obligatorias if col not in dataset.columns]
    
    if columnas_faltantes:
        print(f"⚠️ Faltan variables de salida necesarias en los datos.")
        for col in columnas_faltantes:
            print(f"- No se encontró la columna '{col}'")
        print("El modelo no podrá entrenarse correctamente sin estas variables.")
    
    # Definimos las columnas de características (X) y etiquetas (Y)
    # Filtramos las columnas que existen en el dataset
    X_columns_all = [
        'tiempo_restante', 'puntuacion',
        'aciertos_consecutivos', 'fallos_consecutivos', 'dificultad_actual',
        'tiempo_reaccion_promedio', 'aciertos', 'fallos'
    ]
    
    # Añadimos columnas de estadísticas por agujero
    for i in range(8):  # 8 agujeros
        X_columns_all.extend([
            f'aciertos_agujero_{i}', 
            f'fallos_agujero_{i}', 
            f'tiempo_reaccion_agujero_{i}'
        ])
    
    # Columnas de etiquetas
    y_columns = ['agujero_sugerido', 'velocidad_sugerida']
    
    # Filtramos solo las columnas que existen en el dataset
    X_columns = [col for col in X_columns_all if col in dataset.columns]
    y_columns_filtradas = [col for col in y_columns if col in dataset.columns]
    
    # Verificar si tenemos suficientes características y etiquetas
    if len(X_columns) < 3:
        print(f"⚠️ Solo hay {len(X_columns)} columnas de características disponibles. El modelo puede no ser efectivo.")
        print(f"Columnas disponibles: {X_columns}")
    
    if len(y_columns_filtradas) == 0:
        print("⚠️ No hay columnas de etiquetas disponibles. No se puede entrenar el modelo.")
    else:
        # Eliminar filas con valores nulos o infinitos
        dataset_limpio = dataset.replace([np.inf, -np.inf], np.nan).dropna(subset=X_columns + y_columns_filtradas)
        
        if len(dataset_limpio) < len(dataset):
            print(f"⚠️ Se eliminaron {len(dataset) - len(dataset_limpio)} filas con valores nulos o infinitos.")
        
        if len(dataset_limpio) == 0:
            print("⚠️ No quedan datos después de eliminar valores nulos o infinitos.")
        else:
            # Separamos X e y
            X = dataset_limpio[X_columns]
            y = dataset_limpio[y_columns_filtradas]
            
            # Normalización de características numéricas
            scaler = StandardScaler()
            X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
            
            # División en conjuntos de entrenamiento y prueba
            test_size = 0.2  # 20% para pruebas
            # Usar un tamaño de prueba más pequeño si hay pocos datos
            if len(X) < 50:
                test_size = 0.1  # 10% para pruebas si hay pocos datos
            
            # Si hay muy pocos datos, mostrar una advertencia
            if len(X) < 20:
                print(f"⚠️ Solo hay {len(X)} muestras para entrenamiento y prueba.")
                print("El modelo podría no ser confiable con tan pocos datos.")
                print("Se usará todo el dataset para entrenamiento.")
                X_train, y_train = X_scaled, y
                X_test, y_test = X_scaled.iloc[:2], y.iloc[:2]  # Usar las primeras filas como prueba (solo para estructura)
            else:
                # División normal en entrenamiento y prueba
                X_train, X_test, y_train, y_test = train_test_split(
                    X_scaled, y, test_size=test_size, random_state=42
                )
            
            print(f"✅ Datos divididos en conjuntos de entrenamiento y prueba:")
            print(f"- Conjunto de entrenamiento: {len(X_train)} muestras")
            print(f"- Conjunto de prueba: {len(X_test)} muestras")
            
            # Si hay pocas muestras, usar validación cruzada
            if len(X) < 100:
                print(f"⚠️ Solo hay {len(X)} muestras para entrenamiento. Se recomienda tener al menos 100.")
                print("El modelo puede no ser preciso con tan pocos datos.")

In [ ]:
# Visualizar los conjuntos de entrenamiento y prueba

# Verificar que se hayan creado los conjuntos de datos
if 'X_train' in locals() and 'y_train' in locals() and 'X_test' in locals() and 'y_test' in locals():
    print("Visualizando conjuntos de entrenamiento y prueba...")
    
    # Mostrar información del conjunto de entrenamiento
    print("\n--- Conjunto de entrenamiento ---")
    print(f"Dimensiones de X_train: {X_train.shape}")
    print("Primeras filas de X_train:")
    display(X_train.head())
    
    print(f"\nDimensiones de y_train: {y_train.shape}")
    print("Primeras filas de y_train:")
    display(y_train.head())
    
    # Mostrar información del conjunto de prueba
    print("\n--- Conjunto de prueba ---")
    print(f"Dimensiones de X_test: {X_test.shape}")
    print("Primeras filas de X_test:")
    display(X_test.head())
    
    print(f"\nDimensiones de y_test: {y_test.shape}")
    print("Primeras filas de y_test:")
    display(y_test.head())
    
    # Visualizar la distribución de las variables de salida
    if 'agujero_sugerido' in y_train.columns:
        plt.figure(figsize=(10, 5))
        plt.subplot(1, 2, 1)
        y_train['agujero_sugerido'].value_counts().sort_index().plot(kind='bar')
        plt.title('Distribución de Agujero Sugerido (Train)')
        plt.xlabel('Agujero')
        plt.ylabel('Frecuencia')
        
        plt.subplot(1, 2, 2)
        y_test['agujero_sugerido'].value_counts().sort_index().plot(kind='bar')
        plt.title('Distribución de Agujero Sugerido (Test)')
        plt.xlabel('Agujero')
        plt.ylabel('Frecuencia')
        plt.tight_layout()
        plt.show()
    
    if 'velocidad_sugerida' in y_train.columns:
        plt.figure(figsize=(12, 5))
        plt.subplot(1, 2, 1)
        plt.hist(y_train['velocidad_sugerida'], bins=10)
        plt.title('Distribución de Velocidad Sugerida (Train)')
        plt.xlabel('Velocidad (ms)')
        plt.ylabel('Frecuencia')
        
        plt.subplot(1, 2, 2)
        plt.hist(y_test['velocidad_sugerida'], bins=10)
        plt.title('Distribución de Velocidad Sugerida (Test)')
        plt.xlabel('Velocidad (ms)')
        plt.ylabel('Frecuencia')
        plt.tight_layout()
        plt.show()
else:
    print("⚠️ Los conjuntos de entrenamiento y prueba no se han creado aún.")

## 5. Entrenamiento del Modelo de Deep Learning

En esta sección entrenamos el modelo de deep learning utilizando los datos preprocesados. El modelo tiene las siguientes características:

### Arquitectura del modelo:
- **Tipo**: Red neuronal con múltiples salidas (multi-task learning)
- **Entradas**: Características del estado del juego (aciertos, fallos, tiempos de reacción, etc.)
- **Salidas**:
  1. Predicción del agujero óptimo (clasificación)
  2. Predicción de la velocidad óptima (regresión)
  
### Proceso de entrenamiento:
1. División de datos en conjuntos de entrenamiento y prueba
2. Normalización de características numéricas
3. Entrenamiento con validación cruzada
4. Uso de técnicas para evitar sobreajuste (dropout, early stopping)

### Métricas de evaluación:
- Para la predicción del agujero: Precisión (accuracy)
- Para la predicción de velocidad: Error absoluto medio (MAE)

El número de épocas y el tamaño del batch se ajustan automáticamente según la cantidad de datos disponibles.

In [ ]:
# Verificar que tengamos datos preparados para el entrenamiento
if not all(var in globals() for var in ['X_train', 'y_train', 'X_test', 'y_test']):
    print("⚠️ No hay datos preparados para entrenar el modelo.")
else:
    # Importamos las bibliotecas necesarias para Deep Learning
    import tensorflow as tf
    from tensorflow.keras.models import Model
    from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
    from tensorflow.keras.optimizers import Adam
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    import matplotlib.pyplot as plt
    from sklearn.preprocessing import MinMaxScaler
    import joblib

    # Para reproducibilidad
    tf.random.set_seed(42)

    # Verificamos si hay GPU disponible (opcional)
    print("GPU disponible:", tf.config.list_physical_devices('GPU'))

    # Verificamos si tenemos suficientes datos
    if len(X_train) < 30:
        print(f"⚠️ Solo hay {len(X_train)} muestras para entrenamiento. Se recomienda tener al menos 100.")
        print("El modelo puede no ser preciso con tan pocos datos.")

    # Verificamos si tenemos las variables de salida correctas
    if 'agujero_sugerido' not in y_train.columns or 'velocidad_sugerida' not in y_train.columns:
        print("⚠️ Faltan variables de salida necesarias en los datos.")
        if 'agujero_sugerido' not in y_train.columns:
            print("- No se encontró la columna 'agujero_sugerido'")
        if 'velocidad_sugerida' not in y_train.columns:
            print("- No se encontró la columna 'velocidad_sugerida'")
    else:
        # Definimos la arquitectura del modelo
        def crear_modelo_multi_salida(input_dim):
            # Input layer
            inputs = Input(shape=(input_dim,), name='input_layer')
            
            # Shared layers
            x = Dense(128, activation='relu', name='dense_1')(inputs)
            x = BatchNormalization()(x)
            x = Dropout(0.3)(x)
            
            x = Dense(64, activation='relu', name='dense_2')(x)
            x = BatchNormalization()(x)
            x = Dropout(0.3)(x)
            
            # Branch para predicción de agujero (clasificación)
            agujero_branch = Dense(32, activation='relu', name='agujero_dense')(x)
            agujero_branch = Dropout(0.2)(agujero_branch)
            agujero_output = Dense(8, activation='softmax', name='agujero_output')(agujero_branch)  # 8 agujeros
            
            # Branch para predicción de velocidad (regresión)
            velocidad_branch = Dense(32, activation='relu', name='velocidad_dense')(x)
            velocidad_branch = Dropout(0.2)(velocidad_branch)
            velocidad_output = Dense(1, activation='linear', name='velocidad_output')(velocidad_branch)
            
            # Creamos el modelo con múltiples salidas
            model = Model(inputs=inputs, outputs=[agujero_output, velocidad_output])
            
            # Compilamos el modelo con pérdidas específicas para cada tarea
            model.compile(
                optimizer=Adam(learning_rate=0.001),
                loss={
                    'agujero_output': 'sparse_categorical_crossentropy',
                    'velocidad_output': 'mean_squared_error'
                },
                metrics={
                    'agujero_output': 'accuracy',
                    'velocidad_output': 'mae'
                }
            )
            
            return model
        
        # Creamos el modelo
        input_dim = X_train.shape[1]
        modelo = crear_modelo_multi_salida(input_dim)
        
        # Mostramos un resumen del modelo
        modelo.summary()
        
        # Definimos callbacks para mejorar el entrenamiento
        callbacks = [
            EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
            ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0.0001)
        ]
        
        # Preparamos los datos para el entrenamiento
        X_train_array = X_train.values
        y_train_agujero = y_train['agujero_sugerido'].values
        
        # Escalamos la velocidad sugerida (valores entre 400 y 4000 ms)
        velocidad_scaler = MinMaxScaler(feature_range=(0, 1))
        y_train_velocidad = velocidad_scaler.fit_transform(y_train[['velocidad_sugerida']]).flatten()
            
        X_test_array = X_test.values
        y_test_agujero = y_test['agujero_sugerido'].values
        
        # Aplicamos el mismo escalador a los datos de prueba
        y_test_velocidad = velocidad_scaler.transform(y_test[['velocidad_sugerida']]).flatten()
        
        # Entrenamos el modelo
        print("\nIniciando entrenamiento del modelo...")
        
        # Ajustamos el número de épocas según la cantidad de datos
        epocas = min(100, max(30, 150 - len(X_train) // 10))  # Menos épocas para más datos
        print(f"Entrenando por {epocas} épocas (ajustado según cantidad de datos)")
        
        # Ajustamos el tamaño del batch según la cantidad de datos
        batch_size = min(32, max(8, len(X_train) // 10))
        print(f"Usando batch size de {batch_size} (ajustado según cantidad de datos)")
        
        history = modelo.fit(
            X_train_array, 
            {
                'agujero_output': y_train_agujero,
                'velocidad_output': y_train_velocidad
            },
            epochs=epocas,
            batch_size=batch_size,
            validation_split=0.2,
            callbacks=callbacks,
            verbose=1
        )
        
        # Visualizamos las curvas de aprendizaje
        plt.figure(figsize=(15, 6))
        
        # Gráfico de pérdida
        plt.subplot(1, 2, 1)
        plt.plot(history.history['loss'], label='Train Loss')
        plt.plot(history.history['val_loss'], label='Validation Loss')
        plt.title('Curvas de Pérdida')
        plt.xlabel('Épocas')
        plt.ylabel('Pérdida')
        plt.legend()
        
        # Gráfico de precisión para la predicción de agujero
        plt.subplot(1, 2, 2)
        plt.plot(history.history['agujero_output_accuracy'], label='Train Accuracy')
        plt.plot(history.history['val_agujero_output_accuracy'], label='Validation Accuracy')
        plt.title('Precisión en Predicción de Agujero')
        plt.xlabel('Épocas')
        plt.ylabel('Precisión')
        plt.legend()
        
        plt.tight_layout()
        plt.show()
        
        # Guardar el modelo para uso futuro
        modelo.save('modelo_whacamole_real.h5')
        print("\nModelo guardado como 'modelo_whacamole_real.h5'")
        
        # También guardamos los escaladores para poder normalizar nuevos datos
        if 'scaler' in globals():
            joblib.dump(scaler, 'scaler_features_real.pkl')
            print("Escalador de características guardado como 'scaler_features_real.pkl'")
        joblib.dump(velocidad_scaler, 'scaler_velocidad_real.pkl')
        print("Escalador de velocidad guardado como 'scaler_velocidad_real.pkl'")

## 6. Evaluación del Modelo

Ahora evaluaremos el rendimiento del modelo en el conjunto de prueba para ver qué tan bien generaliza a datos nuevos. Analizaremos tanto la precisión en la predicción del agujero como el error en la predicción de la velocidad.

In [ ]:
# Verificar que el modelo exista
if 'modelo' not in locals():
    print("⚠️ No hay un modelo entrenado para evaluar.")
else:
    # Importar bibliotecas adicionales necesarias
    import numpy as np
    from sklearn.metrics import confusion_matrix, classification_report
    
    # Verificar que existan datos de prueba y las variables necesarias
    if not all(var in globals() for var in ['X_test_array', 'y_test_agujero', 'y_test_velocidad', 'velocidad_scaler']):
        print("⚠️ No se encuentran todas las variables necesarias para la evaluación.")
        print("Asegúrate de haber entrenado el modelo correctamente.")
    else:
        # Evaluamos el modelo en el conjunto de prueba
        print("Evaluando el modelo en el conjunto de prueba...")
        
        try:
            resultados = modelo.evaluate(
                X_test_array,
                {
                    'agujero_output': y_test_agujero,
                    'velocidad_output': y_test_velocidad
                },
                verbose=1
            )
            
            # Mostramos métricas de evaluación
            print(f"\nPérdida total: {resultados[0]:.4f}")
            print(f"Pérdida en predicción de agujero: {resultados[1]:.4f}")
            print(f"Pérdida en predicción de velocidad: {resultados[2]:.4f}")
            print(f"Precisión en predicción de agujero: {resultados[3]:.4f}")
            print(f"Error absoluto medio en predicción de velocidad: {resultados[4]:.4f}")
            
            # Hacemos predicciones en el conjunto de prueba
            y_pred = modelo.predict(X_test_array)
            y_pred_agujero = np.argmax(y_pred[0], axis=1)
            y_pred_velocidad = y_pred[1].flatten()
            
            # Desnormalizamos la velocidad predicha
            try:
                # Convertir predicciones de velocidad a valores reales (ms)
                y_pred_velocidad_real = velocidad_scaler.inverse_transform(y_pred_velocidad.reshape(-1, 1)).flatten()
                y_test_velocidad_real = velocidad_scaler.inverse_transform(y_test_velocidad.reshape(-1, 1)).flatten()
                
                # Matriz de confusión para la predicción de agujero
                cm = confusion_matrix(y_test_agujero, y_pred_agujero)
                
                plt.figure(figsize=(10, 8))
                sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=range(8), yticklabels=range(8))
                plt.title('Matriz de Confusión - Predicción de Agujero')
                plt.xlabel('Predicción')
                plt.ylabel('Real')
                plt.show()
                
                # Reporte de clasificación para la predicción de agujero
                print("\nReporte de clasificación para la predicción de agujero:")
                print(classification_report(y_test_agujero, y_pred_agujero, labels=range(8)))
                
                # Análisis de error en la predicción de velocidad
                plt.figure(figsize=(10, 6))
                plt.scatter(y_test_velocidad_real, y_pred_velocidad_real, alpha=0.5)
                plt.plot([y_test_velocidad_real.min(), y_test_velocidad_real.max()], 
                        [y_test_velocidad_real.min(), y_test_velocidad_real.max()], 
                        'r--')
                plt.title('Velocidad Real vs Predicha')
                plt.xlabel('Velocidad Real (ms)')
                plt.ylabel('Velocidad Predicha (ms)')
                plt.grid(True)
                plt.show()
                
                # Histograma de error en la predicción de velocidad
                error_velocidad = y_test_velocidad_real - y_pred_velocidad_real
                
                plt.figure(figsize=(10, 6))
                plt.hist(error_velocidad, bins=30, alpha=0.7)
                plt.title('Distribución del Error en Predicción de Velocidad')
                plt.xlabel('Error (ms)')
                plt.ylabel('Frecuencia')
                plt.axvline(x=0, color='r', linestyle='--')
                plt.grid(True)
                plt.show()
                
                print(f"\nError medio en predicción de velocidad: {np.mean(error_velocidad):.2f} ms")
                print(f"Error absoluto medio en predicción de velocidad: {np.mean(np.abs(error_velocidad)):.2f} ms")
                print(f"Desviación estándar del error: {np.std(error_velocidad):.2f} ms")
                
                # Mostrar ejemplos de predicciones
                num_ejemplos = min(5, len(X_test))
                print(f"\nEjemplos de predicciones para {num_ejemplos} instancias:")
                for i in range(num_ejemplos):
                    print(f"Ejemplo {i+1}:")
                    print(f"  Agujero real: {y_test_agujero[i]} | Agujero predicho: {y_pred_agujero[i]}")
                    print(f"  Velocidad real: {y_test_velocidad_real[i]:.0f} ms | Velocidad predicha: {y_pred_velocidad_real[i]:.0f} ms")
                
            except Exception as e:
                print(f"Error al evaluar el modelo: {e}")
                print("Es posible que no haya suficientes datos de prueba para una evaluación completa.")
                
        except Exception as e:
            print(f"Error al evaluar el modelo: {e}")
            print("Verifica que el modelo se haya entrenado correctamente y que los datos de prueba sean compatibles.")

## 7. Predicción de Próximo Agujero y Velocidad

Ahora veremos cómo usar el modelo entrenado para hacer predicciones en tiempo real durante el juego. Simularemos una situación de juego y haremos predicciones basadas en el estado actual.

In [ ]:
# Verificar que el modelo exista
if 'modelo' not in locals():
    print("⚠️ No hay un modelo entrenado para hacer predicciones.")
else:
    # Importar bibliotecas adicionales necesarias
    import numpy as np
    import pandas as pd
    
    # Función para hacer predicciones con datos de juego en tiempo real
    def predecir_proximo_agujero_y_velocidad(
        aciertos_por_agujero,
        fallos_por_agujero,
        tiempos_reaccion_por_agujero,
        tiempo_restante,
        puntuacion,
        aciertos_consecutivos,
        fallos_consecutivos,
        dificultad_actual,
        aciertos_totales,
        fallos_totales,
        tiempo_reaccion_promedio
    ):
        # Creamos un DataFrame con los datos de entrada
        datos_entrada = {}
        
        # Añadimos estadísticas básicas
        datos_entrada['tiempo_restante'] = tiempo_restante
        datos_entrada['puntuacion'] = puntuacion
        datos_entrada['aciertos_consecutivos'] = aciertos_consecutivos
        datos_entrada['fallos_consecutivos'] = fallos_consecutivos
        datos_entrada['dificultad_actual'] = dificultad_actual
        datos_entrada['aciertos'] = aciertos_totales
        datos_entrada['fallos'] = fallos_totales
        datos_entrada['tiempo_reaccion_promedio'] = tiempo_reaccion_promedio
        
        # Añadimos estadísticas por agujero
        for i in range(8):  # 8 agujeros
            datos_entrada[f'aciertos_agujero_{i}'] = aciertos_por_agujero[i]
            datos_entrada[f'fallos_agujero_{i}'] = fallos_por_agujero[i]
            datos_entrada[f'tiempo_reaccion_agujero_{i}'] = tiempos_reaccion_por_agujero[i]
        
        # Creamos un DataFrame con una fila
        df_entrada = pd.DataFrame([datos_entrada])
        
        # Verificamos que las columnas coincidan con las esperadas por el modelo
        columnas_faltantes = [col for col in X_train.columns if col not in df_entrada.columns]
        columnas_sobrantes = [col for col in df_entrada.columns if col not in X_train.columns]
        
        if columnas_faltantes:
            print(f"⚠️ Faltan columnas en los datos de entrada: {columnas_faltantes}")
            for col in columnas_faltantes:
                df_entrada[col] = 0  # Añadimos columnas faltantes con valor 0
        
        if columnas_sobrantes:
            print(f"⚠️ Hay columnas sobrantes en los datos de entrada: {columnas_sobrantes}")
            df_entrada = df_entrada.drop(columns=columnas_sobrantes)  # Eliminamos columnas sobrantes
        
        # Reordenamos las columnas para que coincidan con el orden de X_train
        df_entrada = df_entrada[X_train.columns]
        
        # Normalizamos los datos usando el mismo escalador que en el entrenamiento
        if 'scaler' in globals():
            datos_normalizados = scaler.transform(df_entrada)
        else:
            # Si no hay escalador, usamos los datos tal cual
            datos_normalizados = df_entrada.values
        
        # Hacemos la predicción
        predicciones = modelo.predict(datos_normalizados)
        
        # Obtenemos la predicción del agujero (índice con mayor probabilidad)
        prob_agujero = predicciones[0][0]  # Probabilidades para cada agujero
        indice_agujero = np.argmax(prob_agujero)  # Índice del agujero con mayor probabilidad
        
        # Obtenemos la predicción de velocidad y la desnormalizamos
        velocidad_norm = predicciones[1][0][0]  # Valor normalizado
        velocidad_ms = velocidad_scaler.inverse_transform([[velocidad_norm]])[0][0]  # Valor en ms
        
        # Redondeamos la velocidad a un valor entero
        velocidad_ms = int(round(velocidad_ms))
        
        # Limitamos la velocidad a un rango razonable (entre 400 y 4000 ms)
        velocidad_ms = max(400, min(4000, velocidad_ms))
        
        return indice_agujero, velocidad_ms, prob_agujero
    
    # Simulamos una situación de juego para hacer una predicción de ejemplo
    print("Simulando una situación de juego para hacer una predicción...\n")
    
    # Datos de ejemplo
    aciertos_por_agujero = [5, 3, 8, 2, 4, 6, 1, 3]  # Aciertos en cada agujero
    fallos_por_agujero = [2, 1, 0, 4, 3, 1, 5, 2]  # Fallos en cada agujero
    tiempos_reaccion_por_agujero = [800, 950, 750, 1100, 850, 920, 1050, 980]  # Tiempo promedio en ms
    
    tiempo_restante = 45  # Segundos restantes de partida
    puntuacion = 120  # Puntuación actual
    aciertos_consecutivos = 2  # Aciertos consecutivos actuales
    fallos_consecutivos = 0  # Fallos consecutivos actuales
    dificultad_actual = 1200  # Intervalo actual entre topos (ms)
    aciertos_totales = sum(aciertos_por_agujero)  # Total de aciertos
    fallos_totales = sum(fallos_por_agujero)  # Total de fallos
    
    # Calculamos el tiempo de reacción promedio global
    tiempos_validos = [t for i, t in enumerate(tiempos_reaccion_por_agujero) if aciertos_por_agujero[i] > 0]
    tiempo_reaccion_promedio = sum(tiempos_validos) / len(tiempos_validos) if tiempos_validos else 1000
    
    try:
        # Hacemos la predicción
        agujero_predicho, velocidad_predicha, probabilidades = predecir_proximo_agujero_y_velocidad(
            aciertos_por_agujero,
            fallos_por_agujero,
            tiempos_reaccion_por_agujero,
            tiempo_restante,
            puntuacion,
            aciertos_consecutivos,
            fallos_consecutivos,
            dificultad_actual,
            aciertos_totales,
            fallos_totales,
            tiempo_reaccion_promedio
        )
        
        # Mostramos los resultados
        print(f"⭐ Predicción del modelo:")
        print(f"  Agujero sugerido: {agujero_predicho}")
        print(f"  Velocidad sugerida: {velocidad_predicha} ms")
        
        # Mostramos las probabilidades para cada agujero
        print("\nProbabilidades por agujero:")
        for i, prob in enumerate(probabilidades):
            print(f"  Agujero {i}: {prob*100:.2f}%")
        
        # Visualizamos la predicción
        plt.figure(figsize=(10, 6))
        
        # Gráfico de barras para las probabilidades de cada agujero
        plt.subplot(1, 2, 1)
        plt.bar(range(8), probabilidades)
        plt.title('Probabilidad por Agujero')
        plt.xlabel('Agujero')
        plt.ylabel('Probabilidad')
        plt.xticks(range(8))
        
        # Gráfico de comparación de fallos vs predicción
        plt.subplot(1, 2, 2)
        ancho = 0.35
        posiciones = np.arange(8)
        plt.bar(posiciones - ancho/2, fallos_por_agujero, ancho, label='Fallos')
        
        # Resaltamos el agujero predicho
        agujero_destacado = [0] * 8
        agujero_destacado[agujero_predicho] = max(fallos_por_agujero) + 1
        plt.bar(posiciones + ancho/2, agujero_destacado, ancho, label='Agujero Predicho')
        
        plt.title('Comparación: Fallos vs Predicción')
        plt.xlabel('Agujero')
        plt.ylabel('Cantidad')
        plt.xticks(posiciones, range(8))
        plt.legend()
        
        plt.tight_layout()
        plt.show()
        
        print("\n🎮 Interpretación para el juego:")
        print(f"  El modelo sugiere que el próximo topo aparezca en el agujero {agujero_predicho}")
        print(f"  La velocidad recomendada es de {velocidad_predicha} ms entre topos")
        
        # Análisis adicional de la predicción
        agujero_max_fallos = fallos_por_agujero.index(max(fallos_por_agujero))
        if agujero_predicho == agujero_max_fallos:
            print(f"\n✅ La predicción coincide con el agujero donde el usuario tiene más fallos ({agujero_max_fallos})")
        else:
            print(f"\n⚠️ La predicción ({agujero_predicho}) difiere del agujero con más fallos ({agujero_max_fallos})")
            print("  Esto puede deberse a que el modelo considera otros factores además de los fallos")
        
    except Exception as e:
        print(f"Error al hacer la predicción: {e}")
        print("Verifica que el modelo se haya entrenado correctamente y que los datos sean compatibles.")

## 8. Exportación del Modelo para Integración en el Juego

Para usar el modelo en el juego, necesitamos exportarlo a un formato que pueda ser cargado desde JavaScript. Hay dos opciones principales:

1. **Exportar a TensorFlow.js**: Podemos convertir el modelo a formato TensorFlow.js para cargarlo directamente en el navegador.
2. **Crear una API**: Podemos desplegar el modelo como un servicio web y llamarlo desde el juego mediante peticiones HTTP.

En este caso, optaremos por exportar el modelo para TensorFlow.js, ya que permite una experiencia más fluida sin depender de un servidor externo.

In [ ]:
# Verificar que el modelo exista
if 'modelo' not in locals():
    print("⚠️ No hay un modelo entrenado para exportar.")
else:
    # Guardar el modelo en formato H5 (formato nativo de Keras)
    try:
        modelo.save('modelo_whacamole.h5')
        print("Modelo guardado en formato H5")
        
        # También guardamos los escaladores para poder normalizar nuevos datos
        import joblib
        joblib.dump(scaler, 'scaler_features.pkl')
        if 'velocidad_scaler' in locals():
            joblib.dump(velocidad_scaler, 'scaler_velocidad.pkl')
            print("Escaladores guardados")
        
        # Exportar también los parámetros necesarios para la normalización
        import json
        
        # Parámetros del StandardScaler para las características
        scaler_params = {
            'mean': scaler.mean_.tolist(),
            'scale': scaler.scale_.tolist(),
            'var': scaler.var_.tolist()
        }
        
        # Parámetros del MinMaxScaler para la velocidad (si existe)
        if 'velocidad_scaler' in locals():
            velocidad_scaler_params = {
                'min': velocidad_scaler.min_.tolist(),
                'scale': velocidad_scaler.scale_.tolist(),
                'data_min': velocidad_scaler.data_min_.tolist(),
                'data_max': velocidad_scaler.data_max_.tolist(),
                'data_range': velocidad_scaler.data_range_.tolist()
            }
            
            # Guardar los parámetros en archivos JSON
            with open('velocidad_scaler_params.json', 'w') as f:
                json.dump(velocidad_scaler_params, f)
                
        # Guardar los parámetros en archivos JSON
        with open('scaler_params.json', 'w') as f:
            json.dump(scaler_params, f)
        
        print("Parámetros de normalización guardados en formato JSON para uso en JavaScript")
        
        print("""
        Para usar el modelo en JavaScript:
        
        1. Instalar TensorFlow.js:
           npm install @tensorflow/tfjs
        
        2. Cargar el modelo:
           ```javascript
           import * as tf from '@tensorflow/tfjs';
           
           async function cargarModelo() {
             const modelo = await tf.loadLayersModel('modelo_tfjs/model.json');
             return modelo;
           }
           ```
        
        3. Normalizar los datos de entrada:
           ```javascript
           function normalizarDatos(datos, medias, desviaciones) {
             // Aplica la misma normalización que en Python (StandardScaler)
             return datos.map((valor, i) => (valor - medias[i]) / desviaciones[i]);
           }
           ```
        
        4. Hacer predicciones:
           ```javascript
           async function predecirAgujeroYVelocidad(modelo, datosNormalizados) {
             const tensor = tf.tensor2d([datosNormalizados]);
             const predicciones = await modelo.predict(tensor);
             
             // Obtener la predicción del agujero
             const probabilidadesAgujero = await predicciones[0].data();
             const agujeroPredicho = tf.argMax(predicciones[0], 1).dataSync()[0];
             
             // Obtener la predicción de velocidad (y desnormalizar)
             const velocidadNormalizada = await predicciones[1].data();
             const velocidadPredicha = desnormalizarVelocidad(velocidadNormalizada[0]);
             
             return { agujero: agujeroPredicho, velocidad: velocidadPredicha, probabilidades: probabilidadesAgujero };
           }
           ```
        """)
    except Exception as e:
        print(f"Error al exportar el modelo: {e}")
        print("Asegúrate de que el modelo se haya entrenado correctamente.")

## 9. Sugerencias de Cambios en el Código del Juego para Integrar ML

Ya hemos realizado la mayoría de los cambios necesarios en el código JavaScript del juego. A continuación, se resumen los cambios implementados y algunas sugerencias adicionales para mejorar la integración.

### Cambios ya implementados:

1. **Recolección de datos**:
   - Variables para almacenar estadísticas de juego (`mlData`)
   - Registro de aciertos/fallos por agujero
   - Registro de tiempos de reacción
   - Almacenamiento de intentos consecutivos

2. **Predicción de agujero**:
   - Función `predecirMejorAgujero()` con heurística simple
   - Cambio en `randomMole()` para usar predicción después de 10 intentos

3. **Adaptación de velocidad**:
   - Mejora en `ajustarDificultad()` para considerar datos de ML
   - Análisis de intentos recientes y tiempos de reacción

4. **Exportación de datos a Excel (.xls)**:
   - Se implementó la función `exportarDatosAExcel()` que guarda los datos en formato .xls
   - Los archivos se guardan en la raíz del proyecto con el nombre `datos_whacamole_{nickname}_{timestamp}.xls`
   - Cada archivo incluye una hoja de descripción con explicaciones de las columnas

### Flujo de trabajo para entrenar el modelo con datos reales:

1. **Jugar varias partidas**:
   - Juega al menos 3-5 partidas para generar suficientes datos
   - Al finalizar cada partida, se guardará automáticamente un archivo .xls en la raíz del proyecto

2. **Entrenar el modelo**:
   - Ejecuta este notebook, que detectará automáticamente los archivos .xls en la raíz
   - El notebook cargará el archivo más reciente o te permitirá seleccionar uno
   - El modelo se entrenará con los datos reales de tus partidas

3. **Exportar el modelo entrenado**:
   - El modelo entrenado se exportará para ser usado por el juego
   - Los parámetros de normalización también se guardarán para uso en JavaScript

4. **Integrar el modelo en el juego**:
   - Sigue las instrucciones de la sección 8 para integrar TensorFlow.js

### Sugerencias adicionales:

1. **Integrar el modelo TensorFlow.js**:
   ```javascript
   // Al inicio del juego, cargar el modelo
   let modelo;
   let modeloCargado = false;
   
   async function cargarModelo() {
       try {
           modelo = await tf.loadLayersModel('/modelo_tfjs/model.json');
           // Cargar también parámetros de normalización
           const respScaler = await fetch('/scaler_params.json');
           const respVelocidad = await fetch('/velocidad_scaler_params.json');
           
           window.scalerParams = await respScaler.json();
           window.velocidadScalerParams = await respVelocidad.json();
           
           modeloCargado = true;
           console.log('Modelo ML cargado correctamente');
       } catch (error) {
           console.error('Error cargando el modelo:', error);
           // Seguir usando la heurística simple si hay error
       }
   }
   
   // Llamar a esta función al inicio
   cargarModelo();
   ```

2. **Reemplazar la función de predicción**:
   ```javascript
   async function predecirMejorAgujero() {
       // Usar heurística simple si el modelo no está cargado
       if (!modeloCargado) {
           // Código actual de la heurística
           // ...
       } else {
           // Preparar datos de entrada para el modelo
           const datosEntrada = prepararDatosParaModelo();
           
           // Normalizar los datos
           const datosNormalizados = normalizarDatos(datosEntrada, window.scalerParams);
           
           // Hacer predicción con el modelo
           const tensor = tf.tensor2d([datosNormalizados]);
           const predicciones = await modelo.predict(tensor);
           
           // Obtener índice del agujero con mayor probabilidad
           const agujeroPredicho = tf.argMax(predicciones[0], 1).dataSync()[0];
           
           // Actualizar intervalo del topo si es necesario
           const velocidadNormalizada = predicciones[1].dataSync()[0];
           const velocidadSugerida = desnormalizarVelocidad(
               velocidadNormalizada, 
               window.velocidadScalerParams
           );
           
           // Podemos aplicar la velocidad sugerida directamente
           if (Math.abs(velocidadSugerida - moleInterval) > 100) {
               console.log(`🤖 ML sugiere cambiar velocidad a ${velocidadSugerida}ms`);
               moleInterval = velocidadSugerida;
               clearInterval(moleTimerId);
               moleTimerId = setInterval(randomMole, moleInterval);
           }
           
           return agujeroPredicho;
       }
   }
   ```

### Consideraciones finales

- La implementación actual ya es funcional con una heurística simple mientras el modelo se entrena.
- El juego exporta datos a Excel (.xls) al finalizar cada partida, facilitando el entrenamiento.
- Los archivos .xls se guardan en la raíz del proyecto para fácil acceso.
- El modelo se mejorará con más datos de juego, así que juega varias partidas para obtener un modelo más preciso.
- Considera agregar un botón en la interfaz para activar/desactivar el ML si lo deseas.

Con estas mejoras, el juego Whac-a-Mole tendrá un sistema adaptativo basado en machine learning que hará que cada partida sea un desafío personalizado para cada jugador.